# BRSET final-test evaluation

In [1]:
# @@ENV@@ — initialize TensorFlow and import the final-test evaluator
import os
import sys
from pathlib import Path

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
BASE_PATH = Path(os.environ.get("DUM_BASE", os.environ.get("ISIC24_BASE", "/mnt/disk1/home/cgiovannoni/mod-attendance")))
sys.path.insert(0, str(BASE_PATH / "src"))

import tensorflow as tf
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

import pandas as pd
from IPython.display import display
from isic24 import paths as pth
from isic24 import statistical_evaluation as se
from isic24.xai_subset import CENTRAL_SETTINGS

pth.DATASET, pth.VARIANT = "brset", "plain"
ART_PATH = pth.art_path()
OUTPUT_DIR = ART_PATH / "statest"


2026-11-05 16:29:18.401460: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1793892558.417867 2873391 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1793892558.421815 2873391 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
I0000 00:00:1793892561.102679 2873391 gpu_device.cc:2413] Ignoring visible gpu device (device: 1, name: NVIDIA GeForce GT 1030, pci bus id: 0000:65:00.0, compute capability: 6.1) with core count: 3. The minimum required count is 8. You can adjust this requirement with the env var TF_MIN_GPU_MULTIPROCESSOR_COUNT.


In [2]:
# @@PROTOCOL@@ — freeze reference and validation-selected configurations before test inference
STUDIES = {
    "vanilla": "20260909-150705",
    "fixed_scaling": "20260914-094615",
    "learnable_scaling": "20260909-183931",
    "loss_regularization": "20260910-104319",
    "controlled_fusion": "20260911-103310",
    "adaptive_gating": "20260914-235654",
}
SEEDS = [8, 47, 53, 71, 90, 101]
EXPECTED_SETTINGS = {
    "vanilla": 1, "fixed_scaling": 9, "learnable_scaling": 5,
    "loss_regularization": 5, "controlled_fusion": 1, "adaptive_gating": 5,
}
protocol = se.freeze_protocol(
    {name: ART_PATH / "runs" / name / study for name, study in STUDIES.items()},
    ART_PATH, OUTPUT_DIR, SEEDS, EXPECTED_SETTINGS, CENTRAL_SETTINGS,
    n_boot=2000, n_permutations=5000, random_seed=2026,
)
selection = pd.DataFrame(protocol["evaluations"])
display(selection.groupby(["panel", "strategy", "setting_tag", "delta"])
        .agg(n_seeds=("seed", "size"), validation_auc=("selection_val_auc", "mean")))


n_seeds  validation_auc
panel     strategy            setting_tag delta                         
reference adaptive_gating     base        0.0          6        0.957454
          controlled_fusion   base        0.0          6        0.961898
          fixed_scaling       a0p500      0.0          6        0.953889
          image_only_control  a0p000      0.0          6        0.939074
          learnable_scaling   init_a0p500 0.0          6        0.952361
          loss_regularization lam1e-03    0.0          6        0.955463
          table_only_control  a1p000      0.0          6        0.884213
          vanilla             baseline    0.0          6        0.952778
selected  adaptive_gating     ent1e-01    0.0          6        0.964537
          controlled_fusion   base        0.0          6        0.961898
          fixed_scaling       a0p875      0.0          6        0.964722
          image_only_control  a0p000      0.0          6        0.939074
          learnable_scaling   init_a0p900 0.0          6        0.954120
          loss_regularization lam1e-02    0.0          6        0.958148
          table_only_control  a1p000      0.0          6        0.884213
          vanilla             baseline    0.0          6        0.952778

In [3]:
# @@TEST_PREDICTIONS@@ — validate patient separation and predict the untouched test cohort
test_frame, predictions = se.predict_test(protocol, OUTPUT_DIR)
print(f"[TEST] {len(test_frame)} images, {test_frame.patient_id.nunique()} patients")


I0000 00:00:1793892683.888200 2873391 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22800 MB memory:  -> device: 0, name: Quadro RTX 6000, pci bus id: 0000:17:00.0, compute capability: 7.5
I0000 00:00:1793892694.777308 2873391 cuda_dnn.cc:529] Loaded cuDNN version 90300


[TEST] 1/72 adaptive_gating:base:d+0.000:s8
[TEST] 2/72 adaptive_gating:base:d+0.000:s47
[TEST] 3/72 adaptive_gating:base:d+0.000:s53
[TEST] 4/72 adaptive_gating:base:d+0.000:s71
[TEST] 5/72 adaptive_gating:base:d+0.000:s90
[TEST] 6/72 adaptive_gating:base:d+0.000:s101
[TEST] 7/72 controlled_fusion:base:d+0.000:s8
[TEST] 8/72 controlled_fusion:base:d+0.000:s47
[TEST] 9/72 controlled_fusion:base:d+0.000:s53
[TEST] 10/72 controlled_fusion:base:d+0.000:s71
[TEST] 11/72 controlled_fusion:base:d+0.000:s90
[TEST] 12/72 controlled_fusion:base:d+0.000:s101
[TEST] 13/72 fixed_scaling:a0p500:d+0.000:s8
[TEST] 14/72 fixed_scaling:a0p500:d+0.000:s47
[TEST] 15/72 fixed_scaling:a0p500:d+0.000:s53
[TEST] 16/72 fixed_scaling:a0p500:d+0.000:s71
[TEST] 17/72 fixed_scaling:a0p500:d+0.000:s90
[TEST] 18/72 fixed_scaling:a0p500:d+0.000:s101
[TEST] 19/72 fixed_scaling:a0p000:d+0.000:s8
[TEST] 20/72 fixed_scaling:a0p000:d+0.000:s47
[TEST] 21/72 fixed_scaling:a0p000:d+0.000:s53
[TEST] 22/72 fixed_scaling:a0p00

In [4]:
# @@STATISTICS@@ — patient-paired confidence intervals, block-swap tests and seed sign tests
results = se.analyze_test(protocol, test_frame, predictions)


[STATISTICS] Patient bootstrap complete; computing paired contrasts
[STATISTICS] reference: 17 paired comparisons complete
[STATISTICS] selected: 17 paired comparisons complete


In [5]:
# @@RESULTS@@ — inspect predictive estimates and prespecified paired contrasts
display(results["summary"].round(5))
display(results["contrasts"].round(5))
print("Differences = strategy minus baseline: higher AUC and lower Brier/ECE are favorable.")
print("95% intervals cover patient sampling conditional on fitted seeds; seed SD is separate.")
print("Intervals are pointwise; p-values use Holm correction within each panel and test unit.")


,panel,strategy,metric,mean,seed_sd,seed_variance,ci_lo,ci_hi,n_seeds,n_images,n_patients
0,reference,adaptive_gating,auc,0.96086,0.00353,0.00001,0.94603,0.97453,6,3254,1704
1,reference,adaptive_gating,brier,0.02973,0.00544,0.00003,0.02483,0.03489,6,3254,1704
2,reference,adaptive_gating,ece,0.02743,0.01849,0.00034,0.02426,0.03388,6,3254,1704
3,reference,controlled_fusion,auc,0.97212,0.00248,0.00001,0.96141,0.98177,6,3254,1704
4,reference,controlled_fusion,brier,0.02553,0.00539,0.00003,0.02102,0.03029,6,3254,1704
5,reference,controlled_fusion,ece,0.01930,0.01909,0.00036,0.01782,0.02485,6,3254,1704
6,reference,fixed_scaling,auc,0.97318,0.00262,0.00001,0.96235,0.98225,6,3254,1704
7,reference,fixed_scaling,brier,0.02542,0.00339,0.00001,0.02083,0.03025,6,3254,1704
8,reference,fixed_scaling,ece,0.01905,0.01090,0.00012,0.01670,0.02443,6,3254,1704
9,reference,image_only_control,auc,0.95476,0.00222,0.00000,0.93815,0.96996,6,3254,1704


,panel,baseline,strategy,metric,difference,ci_lo,ci_hi,favorable_seeds,n_informative_seeds,patient_swap_p,seed_sign_p,patient_swap_p_holm,seed_sign_p_holm
0,reference,vanilla,adaptive_gating,auc,-0.01099,-0.01879,-0.00379,0,6,0.00040,0.03125,0.0102,1.0
1,reference,vanilla,adaptive_gating,brier,0.00371,0.00168,0.00583,2,6,0.00040,0.68750,0.0102,1.0
2,reference,vanilla,adaptive_gating,ece,0.00456,0.00148,0.00771,3,6,0.00020,1.00000,0.0102,1.0
3,reference,vanilla,controlled_fusion,auc,0.00027,-0.00314,0.00393,3,6,0.89162,1.00000,1.0000,1.0
4,reference,vanilla,controlled_fusion,brier,-0.00049,-0.00163,0.00077,3,6,0.41912,1.00000,1.0000,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,selected,table_only_control,loss_regularization,brier,-0.06660,-0.07413,-0.05955,6,6,0.00020,0.03125,0.0102,1.0
98,selected,table_only_control,loss_regularization,ece,-0.15709,-0.16679,-0.14392,6,6,0.00020,0.03125,0.0102,1.0
99,selected,table_only_control,vanilla,auc,0.08990,0.06376,0.11643,6,6,0.00020,0.03125,0.0102,1.0
100,selected,table_only_control,vanilla,brier,-0.06411,-0.07153,-0.05724,6,6,0.00020,0.03125,0.0102,1.0


Differences = strategy minus baseline: higher AUC and lower Brier/ECE are favorable.
95% intervals cover patient sampling conditional on fitted seeds; seed SD is separate.
Intervals are pointwise; p-values use Holm correction within each panel and test unit.


In [6]:
# @@EXPORT@@ — save final-test tables, provenance and the selected-panel confidence-interval plot
se.save_results(results, protocol, OUTPUT_DIR)
print(f"[SAVED] Final-test results: {OUTPUT_DIR}")


[SAVED] Final-test results: /mnt/disk1/home/cgiovannoni/mod-attendance/artifacts/brset/plain/statest
